In [23]:
import pandas as pd
import numpy as np
from collections import defaultdict
import json

In [26]:
clues = pd.read_csv("clues_clean.csv").drop("Unnamed: 0", axis=1).reset_index()

In [27]:
clues.sample(10)

,index,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10,clue_pct
23035,23035,RECEDES,Dwindles,Ebbs,Ebbs,Falls back,Goes back,Moves away,"Moves back, as a hairline",Subsides,Withdraws,Withdraws,0.625680
29071,29071,TFAL,Big name in cookware,Brand of nonstick cookware,"Cookware brand hidden in ""crestfallen""","Cookware brand that's an anagram of ""flat""",Kitchenware brand with a hyphenated name,Nonstick brand,Nonstick cookware brand,Nonstick cookware brand,Nonstick cookware brand,Purveyor of nonstick cookware,0.185379
12541,12541,HIDALGO,2004 Viggo Mortensen film,2004 Viggo Mortensen movie,Gentleman: Sp.,Horse film of 2004,Horse film of 2004,Spanish gentleman,Spanish nobleman,Spanish nobleman,State north of Mexico City,The Father of Mexican Independence,0.494450
32237,32237,YESMAN,Hardly the assertive type,He's inclined to agree,Stooge,Sycophant,Sycophant,Sycophant,"The animated Smithers, e.g.",Toady,Toady,Unlikely dissenter,0.792056
12771,12771,HONES,Cutting edge creators,Fine-tunes,Fine-tunes,Gets an edge on,Perfects,Perfects,Practices,Sharpens,Sharpens,Works on,0.734142
22436,22436,PURL,Knit alternative,Knit one ____ two,"Knit one, ___ two",Knit's partner,Knitter's direction,Knitter's stitch,Knitting term,Knitting term,Reversal of sorts,Small loop,0.372924
1515,1515,AMUCK,Bad way to run (Var.),Duck ___ (classic Warner Bros. cartoon short),Duck ___ (classic Warner Bros. short),Frenzied,Frenzied: Var.,In a frenzy,In a wild and crazy fashion,One way to run,Out of control,Sort of way to run,0.218686
6954,6954,DEARME,Alas,Good gracious!,Good gracious!,Goodness gracious,Goodness gracious!,Heavens to Betsy!,I do declare!,Oh heavens!,"Oh, what am I to do?",Worrywart's words,0.918282
11970,11970,HADATIT,Battled,Came to blows,Didn't shrink from the challenge,Fought,Fought,Joined the fight,Joined the fray,Locked horns,Scrapped,Started fighting,0.059630
19190,19190,NSA,Code-breaking govt. group,Counterintelligence org.,Cryptanalysis org.,Grp. concerned with SIGINT,Hush-hush org.,Part of the Defense Dept.,Surveillance org.,Tapping grp.,Ultrasecret org.,Ultrasecret org.,0.867491


In [28]:
clues["length"]=[len(x) for x in clues.answer]

In [29]:
clues_by_length = {l:t for l,t in clues.groupby("length")}

In [30]:
clues_by_length[3]

,index,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10,clue_pct,length
0,0,AAA,Bumper-sticker letters,Grp. that offers International Driving Permits,Minor-league classification,Motorist's org.,Org. with towers,Roadside assistance grp.,Roadside assistance org.,Small battery,"Strong, as a bond",Top bond rating,0.893993,3
4,4,AAH,Circus exclamation,Foot rub reaction,Ooh and ___,Open wide sound,Reaction to a cold drink on a hot day,Sound of wonderment,That hit the spot!,That's nice,What a relief!,Word to a doctor,0.820230,3
13,13,AAR,Interlaken river,River flowing into the Rhine,River in Switzerland,Swiss river,Swiss river,Swiss river,Swiss river,Swiss river,Swiss river into the Rhine,Swiss river to the Rhine,0.522527,3
19,19,AAS,Batteries for some remotes,Furby batteries,Remote batteries,Small batteries,Small batteries,Smallish batteries,"Smoke detector batteries, often",Some batteries,Some remote power sources,___ in Able,0.678004,3
20,20,AAU,Amateur sports org.,Jr. Olympics sponsor,Junior Olympics org.,Org. that awards the Sullivan Award,Org. that bestows the Sullivan Award,"Org. with the Sullivan Award for character, le...","Org. with the Sullivan Award for character, le...",Org. with the annual Junior Olympic Games,"Sports For All, Forever org.",Sports org. for nonprofessionals,0.166078,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32443,32443,ZIT,Bit of acne,Clearasil target,Clearasil target,Minor blemish,Pimple,Proactiv target,Single outbreak,Teen's woe,"Temple spot, say?","Unwanted blemish, informally",0.442580,3
32446,32446,ZOA,Animal group suffix,Animal group suffix,"Animals, in taxonomy",Animals: Suffix,Animals: Suffix,"Combining form meaning ""animals""",Ending with proto-,Organisms: Suffix,Suffix with proto-,Taxonomy suffix,0.071555,3
32448,32448,ZOE,Actress Caldwell,Actress/model Kravitz,"Kravitz of ""Big Little Lies""",Muppet in a tutu,Playwright Akins,Pulitzer playwright Akins,Pulitzer winner Akins,"Saldana of ""Avatar""","Saldana of ""Guardians of the Galaxy""",Saldana who plays Uhura,0.477473,3
32456,32456,ZOO,Chaotic place,"Dr. Seuss' ""If I Ran the ___""","Madhouse, so to speak",Menagerie,Morning ___ (radio format),Petting setting,Place to put stock in?,Place to see pandas and panthers,Place with moving exhibits,San Diego attraction,0.641343,3


In [31]:
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [32]:
character_set = list(alphabet)+[alphabet[i]+alphabet[j] for i in range(26) for j in range(i,26)]

In [33]:
len(character_set)

377

In [34]:
character_set[-1]

'ZZ'

In [35]:
char_table = {ch:n for n,ch in enumerate(character_set)}
idx_table = {n:ch for n,ch in enumerate(character_set)}

In [36]:
def decompose_word(w):
    if len(w) == 0:
        return []
    elif len(w) == 1:
        return [[w]]
    elif len(w) == 2:
        return [[w[0],w[1]],["".join(sorted(w))]]
    else:
        out = []
        first = w[0]
        first_suffix = decompose_word(w[1:])
        for i in first_suffix:
            out.append([first]+i)
        pair = "".join(sorted(w[:2]))
        second_suffix = decompose_word(w[2:])
        for i in second_suffix:
            out.append([pair]+i)
        return out

In [37]:
def decomp_to_idx(l):
    return tuple(char_table[i] for i in l)

In [38]:
decompose_word("ZYDECO")

[['Z', 'Y', 'D', 'E', 'C', 'O'],
 ['Z', 'Y', 'D', 'E', 'CO'],
 ['Z', 'Y', 'D', 'CE', 'O'],
 ['Z', 'Y', 'DE', 'C', 'O'],
 ['Z', 'Y', 'DE', 'CO'],
 ['Z', 'DY', 'E', 'C', 'O'],
 ['Z', 'DY', 'E', 'CO'],
 ['Z', 'DY', 'CE', 'O'],
 ['YZ', 'D', 'E', 'C', 'O'],
 ['YZ', 'D', 'E', 'CO'],
 ['YZ', 'D', 'CE', 'O'],
 ['YZ', 'DE', 'C', 'O'],
 ['YZ', 'DE', 'CO']]

In [39]:
[decomp_to_idx(x) for x in decompose_word("ZYDECO")]

[(25, 24, 3, 4, 2, 14),
 (25, 24, 3, 4, 89),
 (25, 24, 3, 79, 14),
 (25, 24, 102, 2, 14),
 (25, 24, 102, 89),
 (25, 122, 4, 2, 14),
 (25, 122, 4, 89),
 (25, 122, 79, 14),
 (375, 3, 4, 2, 14),
 (375, 3, 4, 89),
 (375, 3, 79, 14),
 (375, 102, 2, 14),
 (375, 102, 89)]

In [40]:
tags_to_word_index =[]
tags_by_length = defaultdict(list)
checked_tags = set()
for i, w, p in zip(clues["index"], clues["answer"], clues["clue_pct"]):
    d = decompose_word(w)
    for t in d:
        tags = decomp_to_idx(t)
        tags_to_word_index.append((i,tags))
        tags_by_length[len(tags)].append((tags,p))
        

In [44]:
len(tags_by_length)

14

In [45]:
with open("tags_by_length.json", "w") as f:
    json.dump(tags_by_length,f, indent = 3)

In [46]:
with open("tags_to_word_index.json","w") as f:
    json.dump(tags_to_word_index, f, indent=3)